<a href="https://colab.research.google.com/github/KULDEEPSONI-source/MACHINE-LEARNING/blob/main/%20kNN%20Classifier%20and%20Decision%20Tree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
df=pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

In [4]:
df.head(5)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [42]:
df.TotalCharges = pd.to_numeric(df.TotalCharges, errors='coerce')

The Role of errors='coerce'
In real-world datasets, columns that should be numeric often contain "dirty" data—such as empty spaces, strings (like "N/A"), or accidental characters.

Without errors='coerce': If pandas encounters a value it cannot turn into a number (like the string "ten"), it will throw a ValueError and stop execution entirely.

With errors='coerce': This tells pandas: "If you find a value that isn't a number, don't crash; just turn that specific value into a NaN (Not a Number/Missing Value) instead."

In [43]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [45]:
df.dropna(how='any', inplace=True)

The Role of inplace=True
When you set inplace=True, you are telling pandas to modify the original DataFrame object directly rather than creating a new copy.

inplace=True: The operation happens on the existing df. The method returns None.

inplace=False (Default): The operation creates a copy of your DataFrame, performs the change on that copy, and returns the new version. Your original df remains unchanged.

In [66]:
df.Churn.value_counts()/len(df)*100

,count
Churn,
No,73.421502
Yes,26.578498


In [67]:
x=df.drop(['customerID', 'Churn'], axis=1)

In [68]:
y=df.Churn.values

In [69]:
x

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,Male,0,Yes,Yes,24,Yes,Yes,DSL,Yes,No,Yes,Yes,Yes,Yes,One year,Yes,Mailed check,84.80,1990.50
7039,Female,0,Yes,Yes,72,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,One year,Yes,Credit card (automatic),103.20,7362.90
7040,Female,0,Yes,Yes,11,No,No phone service,DSL,Yes,No,No,No,No,No,Month-to-month,Yes,Electronic check,29.60,346.45
7041,Male,1,Yes,No,4,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Mailed check,74.40,306.60


In [52]:
y

array(['No', 'No', 'Yes', ..., 'No', 'Yes', 'No'], dtype=object)

In [70]:
x.columns

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges'],
      dtype='object')

In [88]:
# Convert categorical features to numericals --> Feature Encoding --> Dummy Encoding

X = pd.get_dummies(x, columns=['gender', 'Partner', 'Dependents',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'], drop_first=True)


In [90]:
X.head(1)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Female,gender_Male,Partner_No,Partner_Yes,Dependents_No,Dependents_Yes,...,StreamingMovies_Yes,Contract_Month-to-month,Contract_One year,Contract_Two year,PaperlessBilling_No,PaperlessBilling_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,True,False,False,True,True,False,...,False,True,False,False,False,True,False,False,True,False


In [59]:
# Splitting the data into training & test

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.25)

In [60]:
len(X_train)

5274

In [61]:
len(X_test)

1758

In [62]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
X_train_sc = sc.fit_transform(X_train)
X_test_sc = sc.transform(X_test)

In [63]:
X_train_sc

array([[-0.4377158 , -0.05775315, -0.15578334, ..., -0.52806769,
         1.40461472, -0.5441889 ],
       [-0.4377158 , -1.03779905, -1.3701005 , ..., -0.52806769,
        -0.711939  ,  1.8375972 ],
       [-0.4377158 , -1.16030479, -0.20408951, ..., -0.52806769,
        -0.711939  ,  1.8375972 ],
       ...,
       [-0.4377158 ,  1.61649193,  1.20844952, ..., -0.52806769,
        -0.711939  , -0.5441889 ],
       [-0.4377158 , -0.30276462, -0.88370734, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       [-0.4377158 , -0.87445807,  0.19568569, ..., -0.52806769,
        -0.711939  ,  1.8375972 ]])

In [64]:
X_test_sc

array([[-0.4377158 , -0.67028184,  0.67541592, ..., -0.52806769,
         1.40461472, -0.5441889 ],
       [-0.4377158 ,  0.71811652,  0.19568569, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       [ 2.28458741,  1.53482144,  0.79701421, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       ...,
       [ 2.28458741, -1.16030479,  1.17180346, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       [-0.4377158 , -1.11946954, -0.46894058, ..., -0.52806769,
        -0.711939  , -0.5441889 ],
       [-0.4377158 ,  0.06475259, -0.65883379, ..., -0.52806769,
         1.40461472, -0.5441889 ]])

KNN CLASSIFIER

In [73]:
# Call the kNN Classifier
from sklearn.neighbors import KNeighborsClassifier

# Initiating the classifier
model = KNeighborsClassifier()

# Passing the data to classifier
model.fit(X_train_sc, y_train)

KNeighborsClassifier()

In [74]:
y_pred = model.predict(X_test_sc)

In [75]:
y_pred

array(['No', 'No', 'No', ..., 'No', 'No', 'No'], dtype=object)

In [76]:
y_test

array(['No', 'No', 'No', ..., 'Yes', 'No', 'No'], dtype=object)

In [77]:
# Classification metrics = to check how the model is behaving

from sklearn.metrics import accuracy_score

print(accuracy_score(y_test,y_pred)*100)

76.39362912400455


new prdiction

In [78]:
X_test_sc

array([[-0.4377158 , -0.67028184,  0.67541592, ..., -0.52806769,
         1.40461472, -0.5441889 ],
       [-0.4377158 ,  0.71811652,  0.19568569, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       [ 2.28458741,  1.53482144,  0.79701421, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       ...,
       [ 2.28458741, -1.16030479,  1.17180346, ...,  1.89369664,
        -0.711939  , -0.5441889 ],
       [-0.4377158 , -1.11946954, -0.46894058, ..., -0.52806769,
        -0.711939  , -0.5441889 ],
       [-0.4377158 ,  0.06475259, -0.65883379, ..., -0.52806769,
         1.40461472, -0.5441889 ]])

In [79]:
X_test.columns

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes',
       'MultipleLines_No phone service', 'MultipleLines_Yes',
       'InternetService_Fiber optic', 'InternetService_No',
       'OnlineSecurity_No internet service', 'OnlineSecurity_Yes',
       'OnlineBackup_No internet service', 'OnlineBackup_Yes',
       'DeviceProtection_No internet service', 'DeviceProtection_Yes',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='object')

In [80]:
data = [[0, 2, 87, 178, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1]]

In [81]:
data_sc = sc.transform(data)

single = model.predict(data_sc)

print(single)

['Yes']


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


decision tree classification

In [82]:
# Call the DT Classifier
from sklearn.tree import DecisionTreeClassifier

# Initiating the classifier
model_dt = DecisionTreeClassifier()

# Passing the data to classifier
model_dt.fit(X_train_sc, y_train)

DecisionTreeClassifier()

In [83]:
y_pred_dt = model_dt.predict(X_test_sc)

In [84]:
# Classification metrics = to check how the model is behaving

from sklearn.metrics import accuracy_score

print(accuracy_score(y_test,y_pred_dt)*100)

73.54948805460751
